<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 27 — SVM והשוואת מודלים
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
כמה קווים יכולים להפריד בין דוגמאות. SVM מציע רעיון של שוליים ופשרה בין רוחב ההפרדה לבין מחיר חריגות.

## מטרות
- להסביר שוליים.
- לפרש ציון חתום.
- להשוות מודלים בהוגנות.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## הנחיות למורה
זהו פתרון מתועד למעבדה. מבדילים בין תוצאה נכונה לבין נימוק נכון, ובודקים שהקוד תואם לנתוני התלמיד. המקורות הישנים נשמרו ללא שינוי.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## פתרון חלק 1
הפתרון הבא תואם לנתוני התרגול העצמאי, ולא לנתוני הדוגמה הפתורה.

</div>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report,confusion_matrix
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df=pd.read_csv(url)
model_df=df.dropna(subset=["sex"]).copy()
features=["bill_depth_mm","body_mass_g"]
X=model_df[features].copy();y=model_df["sex"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=17,stratify=y)

<div dir="rtl" lang="he" style="text-align: right;">

## Pipeline ודוח
השורות בעלות sex ידוע בלבד נכנסות. אין השלמת y. מדווחים support וטעויות גם אם הציון גבוה.

</div>

In [ ]:
# kernel ו-C נקבעו מראש; אין חיפוש על Test
model=make_pipeline(
 SimpleImputer(strategy="mean"),StandardScaler(),
 SVC(kernel="linear",C=1.0)
)
model.fit(X_train,y_train)
predictions=model.predict(X_test)
labels=["female","male"]
matrix=confusion_matrix(y_test,predictions,labels=labels)
display(pd.DataFrame(matrix,index=labels,columns=labels))
print(classification_report(y_test,predictions,labels=labels,zero_division=0))
print("Classes:",model.named_steps["svc"].classes_)
print("Number of support vectors:",model.named_steps["svc"].n_support_)

<div dir="rtl" lang="he" style="text-align: right;">

## ציון חתום
C גדול יותר אינו 'מספר שכנים גדול' ואינו ערובה לשיפור. משווים מועמדים ב־ CV על אימון בלבד; כלל זה חל גם על בחירת kernel.

</div>

In [ ]:
decision_scores=model.decision_function(X_test)
print(decision_scores[:5],model.named_steps["svc"].classes_)

<div dir="rtl" lang="he" style="text-align: right;">

## פירוש ומגבלה
לא מעתיקים יעד species כשהעוגן הוא sex; לא מפרשים kernel כקידוד קטגוריות; לא מבטיחים שוליים ללא הפרות בכל נתון. למתקשים מקבלים Pipeline נתון ומסבירים שלבים, למצטיינים מציעים תכנית CV כתובה בלבד.
המספרים נקראים מפלט ההרצה; אין לדרוש ציון ביצועים מסוים. הסבר חלופי מנומק יכול להיות נכון.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## תשובה לכרטיס היציאה
לא. זהו ציון החלטה. אין להסיק ממנו הסתברות בלי הליך ייעודי ותנאים נוספים.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## משוב דידקטי
בדקו בנפרד: בחירת פעולה, תקינות קוד, פירוש פלט, יחידות ומגבלה. שאלו מה ישתנה בקלט חדש. אם יש קושי, חזרו לדוגמה קטנה לפני הגדלת המאגר. זהו משוב מקומי, לא מחוון ניקוד רשמי.

</div>

## בדיקות המורה — מעבדת SVM על sex
למעבדה שני מאפיינים בלבד, בניגוד לארבעה בדוגמת המורה. מציבים ערכי C ו־kernel מראש. בדיקת Test אינה משמשת לבחירת מודל.

In [ ]:
assert features==["bill_depth_mm","body_mass_g"]
assert X_train.shape[1]==2
assert set(X_train.index).isdisjoint(set(X_test.index))
assert y_train.index.equals(X_train.index)
assert y_test.index.equals(X_test.index)
assert model.named_steps["svc"].kernel=="linear"
assert model.named_steps["svc"].C==1.
assert list(model.named_steps["svc"].classes_)==["female","male"]
assert matrix.shape==(2,2) and matrix.sum()==len(y_test)
assert np.array_equal(matrix,confusion_matrix(y_test,predictions,labels=labels))
assert len(decision_scores)==len(predictions)
assert np.isfinite(decision_scores).all()
assert np.array_equal(np.where(decision_scores>=0,model.named_steps["svc"].classes_[1],model.named_steps["svc"].classes_[0]),predictions)
print("Teacher SVM checks passed")

## משוב למורה
חובה להסביר מה נחשב שגיאת סיווג במטריצה, מהו ציון חתום ומה הוא **לא**, ומדוע אי אפשר להסיק מהתאמה על Test שיש להחליף C לאחר הצצה בתוצאות. יש להבחין בין הצלחה על Test נתון לבין הכללה באוכלוסייה.